<a href="https://colab.research.google.com/github/sergi-villanueva/ChatBot-LanParty/blob/main/xatbot2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install -U flask-cors pyngrok google-genai

import os
import json
import threading
from pathlib import Path

from flask import Flask, request, jsonify, send_from_directory
from flask_cors import CORS
from google import genai
from google.genai import types
from google.colab import userdata
from pyngrok import ngrok

# ==============================
# CONFIGURACIÓN
# ==============================

GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
if not GOOGLE_API_KEY:
    raise ValueError("No s'ha trobat la clau API. Afegeix GOOGLE_API_KEY als secrets de Colab.")

token = userdata.get("token_ngrok")
if not token:
    raise ValueError("No s'ha trobat el secret token_ngrok")

ngrok.set_auth_token(token.strip())

client = genai.Client(api_key=GOOGLE_API_KEY)

system_instruction = """
Ets un assistent esportiu dissenyat per a proporcionar rutines d'entrenament a usuaris d'una aplicació.
A partir de les dades que rebis (edat, pes, sexe, disponibilitat de temps, objectius, punt de partida, lesions, etc.)
elabora respostes clares, concretes i útils.
"""

# ==============================
# CARGA DE FAQS
# ==============================

def carregar_faqs():
    ruta = Path("faqs.json")
    if not ruta.exists():
        raise FileNotFoundError("No s'ha trobat faqs.json al directori actual.")
    with open(ruta, "r", encoding="utf-8") as f:
        return json.load(f)

def convertir_faqs_a_text(faqs):
    parts = []
    for i, item in enumerate(faqs, 1):
        pregunta = item.get("pregunta") or item.get("q") or item.get("question") or ""
        resposta = item.get("resposta") or item.get("a") or item.get("answer") or ""
        parts.append(f"{i}. P: {pregunta}\n   R: {resposta}")
    return "\n\n".join(parts)

faqs = carregar_faqs()
faq_text = convertir_faqs_a_text(faqs)

chat = client.chats.create(
    model="gemini-2.5-flash",
    config=types.GenerateContentConfig(
        system_instruction=system_instruction + "\n\nFAQs disponibles:\n" + faq_text,
        temperature=0.7,
        max_output_tokens=1000,
    ),
)

# ==============================
# FLASK APP
# ==============================

app = Flask(__name__, static_folder=".", static_url_path="")
CORS(app)

@app.route("/")
def home_page():
    return send_from_directory(".", "widget.html")

@app.route("/chat", methods=["POST"])
def chat_endpoint():
    data = request.get_json(silent=True) or {}
    prompt = (data.get("message") or "").strip()

    if not prompt:
        return jsonify({"error": "Falta el camp 'message'"}), 400

    try:
        resposta = chat.send_message(prompt)
        return jsonify({"response": resposta.text.strip()})
    except Exception as e:
        return jsonify({"error": str(e)}), 500

# ==============================
# ARRANQUE DEL SERVIDOR
# ==============================

def run_app():
    app.run(host="0.0.0.0", port=5000, debug=False, use_reloader=False)

threading.Thread(target=run_app, daemon=True).start()

public_url = ngrok.connect(5000)
print("URL pública:", public_url)

FileNotFoundError: No s'ha trobat faqs.json al directori actual.